# 🛡️ Exploratory Data Analysis
## Image-Based Inappropriate Content Detection

This notebook performs exploratory analysis of the dataset before training.

**Contents:**
1. Dataset Overview
2. Class Distribution
3. Sample Image Visualization
4. Image Statistics (size, channels, pixel distribution)
5. Augmentation Preview
6. Model Architecture Summary

In [ ]:
import os
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import tensorflow as tf
from tensorflow import keras

from src.dataset import DatasetLoader, CLASS_NAMES, build_augmentation_layer
from src.model import ContentModerationModel
from src.utils import get_class_color, plot_sample_images

# Plotting config
plt.rcParams['figure.facecolor'] = '#1a1a2e'
plt.rcParams['axes.facecolor'] = '#1a1a2e'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.color'] = 'white'

print('TensorFlow version:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))

## 1. Dataset Overview

In [ ]:
DATASET_DIR = PROJECT_ROOT / 'dataset'

# Count files per split and class
splits = ['train', 'validation', 'test']
class_names = CLASS_NAMES

summary = {}
for split in splits:
    summary[split] = {}
    for cls in class_names:
        class_dir = DATASET_DIR / split / cls
        if class_dir.exists():
            count = len(list(class_dir.glob('*.jpg'))) + \
                    len(list(class_dir.glob('*.jpeg'))) + \
                    len(list(class_dir.glob('*.png')))
            summary[split][cls] = count
        else:
            summary[split][cls] = 0

import pandas as pd
df_summary = pd.DataFrame(summary).T
df_summary['Total'] = df_summary.sum(axis=1)
print('\nDataset Summary:')
print(df_summary.to_string())

## 2. Class Distribution

In [ ]:
# Load datasets
loader = DatasetLoader(dataset_dir=str(DATASET_DIR), batch_size=32)
train_ds, val_ds, test_ds = loader.get_datasets()
distribution = loader.get_class_distribution()
class_weights = loader.compute_class_weights()

print('Class Distribution (train):', distribution)
print('Class Weights:', class_weights)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#1a1a2e')

colors = [get_class_color(c) for c in class_names]

# Bar chart
axes[0].set_facecolor('#1a1a2e')
bars = axes[0].bar(class_names, list(distribution.values()), color=colors,
                   edgecolor='white', alpha=0.85, width=0.6)
axes[0].set_title('Training Class Distribution', color='white', fontsize=13)
axes[0].set_ylabel('Sample Count', color='white')
for bar, count in zip(bars, distribution.values()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 str(count), ha='center', color='white', fontsize=11)

# Pie chart
axes[1].set_facecolor('#1a1a2e')
wedges, texts, autotexts = axes[1].pie(
    list(distribution.values()),
    labels=class_names,
    colors=colors,
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': '#1a1a2e', 'linewidth': 2}
)
for t in texts + autotexts:
    t.set_color('white')
axes[1].set_title('Class Distribution (Pie)', color='white', fontsize=13)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'models' / 'class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Sample Image Visualization

In [ ]:
images, labels = loader.get_sample_images(split='train', n_samples=9)
fig = plot_sample_images(images, labels, class_names=CLASS_NAMES, n_cols=3)
plt.show()

## 4. Image Statistics

In [ ]:
# Pixel intensity distribution per class
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.patch.set_facecolor('#1a1a2e')

for i, (img, label) in enumerate(zip(images[:3], labels[:3])):
    ax = axes[i]
    ax.set_facecolor('#1a1a2e')
    color = get_class_color(CLASS_NAMES[label])
    
    channel_names = ['Red', 'Green', 'Blue']
    channel_colors = ['#e74c3c', '#2ecc71', '#3498db']
    
    for ch, (ch_name, ch_color) in enumerate(zip(channel_names, channel_colors)):
        ax.hist(img[:, :, ch].ravel(), bins=50, alpha=0.6,
                color=ch_color, label=ch_name)
    
    ax.set_title(f'Sample {i+1}: {CLASS_NAMES[label]}', color=color, fontsize=11)
    ax.set_xlabel('Pixel Value', color='white')
    ax.set_ylabel('Frequency', color='white')
    ax.legend(facecolor='#2c2c54', labelcolor='white')

fig.suptitle('RGB Channel Distributions', color='white', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 5. Augmentation Preview

In [ ]:
# Show augmented versions of a single image
sample_img = images[0:1]  # (1, H, W, 3)
aug_layer = build_augmentation_layer(seed=None)

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
fig.patch.set_facecolor('#1a1a2e')

axes[0, 0].imshow(np.clip(sample_img[0], 0, 1))
axes[0, 0].set_title('Original', color='#ccd6f6', fontsize=10)
axes[0, 0].axis('off')

for idx in range(1, 10):
    row, col = divmod(idx, 5)
    aug_img = aug_layer(sample_img, training=True)[0]
    axes[row, col].imshow(np.clip(aug_img.numpy(), 0, 1))
    axes[row, col].set_title(f'Augmented {idx}', color='#8892b0', fontsize=9)
    axes[row, col].axis('off')

fig.suptitle('Data Augmentation Preview', color='white', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Model Architecture Summary

In [ ]:
model_wrapper = ContentModerationModel()
model = model_wrapper.build_model()
model_wrapper.compile_model()

print('\n=== Model Architecture ===')
model_wrapper.summary()

params = model_wrapper.get_trainable_params()
print('\n=== Parameter Count ===')
for k, v in params.items():
    print(f'  {k:<20}: {v:>12,}')

In [ ]:
# Visualise MobileNetV2 base layer names (last 30 = fine-tuning candidates)
base = model_wrapper.get_base_model()
layers = base.layers
print(f'Total MobileNetV2 layers: {len(layers)}')
print(f'\nLast 30 layers (will be unfrozen during fine-tuning):')
for layer in layers[-30:]:
    print(f'  [{"BN" if "batch_norm" in layer.name else "  "}] {layer.name}')